# 06 · Monitor chiller status and temperature

Watch one completed poll at a time: UTC, liquid temperature, target, operating mode, chiller/dryer state, summary alarm and named active alarm flags. This example creates no CSV and enables no target, mode or sensor-selection writes. Connecting still sends U and can change AUTO START to RUN.


## 1. Environment and imports

Run `uv sync --locked --group notebooks` from the repository root and select the installed kernel. Restart the kernel after changing driver code. Run one device owner at a time.


In [ ]:
import sys
from pathlib import Path

# Prefer checkout sources when launched from the repository or examples folder.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [ ]:
from math import isfinite
from time import monotonic, sleep

from tark_chiller.controller import TarkChiller
from tark_chiller.protocols import T255P

## 2. Choose the port and monitoring duration

Verify the selected serial adapter. The default interval is one second for 60 seconds. Set `DURATION_S = None` to keep monitoring until you interrupt the cell. Interrupting releases software resources; it does not request equipment standby.


In [ ]:
PORT = "COM9"  # Verify the selected T255P serial adapter.
DURATION_S = 60.0  # None: continue until Interrupt Kernel.
INTERVAL_S = 1.0

## 3. Connection and displayed fields

The operation cell below owns the complete connection with 9600/8N1/XON-XOFF and verified adapter RTS/DTR levels. Prepare the filled loop before running it. The worker coordinates U/J/I/H0; `snapshot()` reads memory only. The formatter labels failed polls UNAVAILABLE instead of presenting partial or missing temperature as live.


In [ ]:
def print_sample(sample):
    """Print one completed poll; failed readings are never presented as live."""
    stamp = sample["timestamp_utc"].isoformat(timespec="seconds")
    status = sample["status"]
    if sample["error"] or not status["connected"]:
        print(f"{stamp} | UNAVAILABLE | {sample['error'] or status['detail']}", flush=True)
        return
    active = ", ".join(name for name, value in status["alarms"].items() if value) or "none"
    print(
        f"{stamp} | {sample['temperature_c']:.2f} C | "
        f"target={sample['setpoint_c']:.2f} C | mode={status['mode']} | "
        f"chiller={'On' if status['chiller_on'] else 'Off'} | "
        f"dryer={'On' if status['dryer_on'] else 'Off'} | "
        f"summary_alarm={status['summary_alarm']} | active alarms: {active}",
        flush=True,
    )

## 4. Monitor live readings

Each new completed poll prints one line. Mode and alarm flags use structured fields rather than parsing diagnostic text. Sensor faults and communication errors stop this example and remain visible. Normal duration expiry stops monitoring; a final in-flight poll is displayed before cleanup. No CSV, extra browser or dashboard is needed.


In [ ]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')
if DURATION_S is not None and (not isfinite(DURATION_S) or DURATION_S <= 0):
    raise ValueError("DURATION_S must be positive finite seconds or None")
if not isfinite(INTERVAL_S) or INTERVAL_S <= 0:
    raise ValueError("INTERVAL_S must be positive finite seconds")

chiller = TarkChiller(
    {
        "port": PORT,
        "baudrate": 9600,
        "bytesize": 8,
        "parity": "N",
        "stopbits": 1,
        "xonxoff": True,
        "rtscts": False,
        "dsrdtr": False,
        "rts": False,
        "dtr": False,
    },
    protocol=T255P(allow_writes=False),
    setpoint_range=(5.0, 45.0),
    coolant_source="ThermoTek T255P Rev C pp9/20; verify installed distilled water",
)

with chiller:
    # The worker coordinates U/J/I/H0. Reading snapshot() adds no serial queries.
    # No target, mode or sensor-selection writes are enabled; no CSV is created.
    chiller.start_monitoring(interval_s=INTERVAL_S, history_size=300)
    deadline = None if DURATION_S is None else monotonic() + DURATION_S
    last_reported = float("-inf")
    print("Live monitoring: UTC | temperature | target | mode | on/off | alarms", flush=True)
    while True:
        snapshot = chiller.snapshot()
        for sample in snapshot["history"]:
            if sample["completed_monotonic"] > last_reported:
                print_sample(sample)
                last_reported = sample["completed_monotonic"]
        if snapshot["service_error"] or snapshot["failed_samples"] or not snapshot["running"]:
            raise RuntimeError(
                snapshot["service_error"] or "A monitoring poll failed; inspect the reported error"
            )
        if deadline is not None and monotonic() >= deadline:
            break
        sleep(min(0.1, INTERVAL_S))

    chiller.stop_monitoring()
    snapshot = chiller.snapshot()
    # Include a final poll that completed while stop_monitoring() was waiting.
    for sample in snapshot["history"]:
        if sample["completed_monotonic"] > last_reported:
            print_sample(sample)
            last_reported = sample["completed_monotonic"]
    if snapshot["service_error"] or snapshot["failed_samples"]:
        raise RuntimeError(snapshot["service_error"] or "A final monitoring poll failed")
    print(f"Monitoring complete: {snapshot['sample_count']} samples; no CSV written.")

# Duration expiry or a kernel interrupt releases the worker and serial port.
# The equipment retains its mode and target; disconnect is not a stop command.
print("Port closed. Follow the equipment procedure separately to stop the chiller/pump.")

## 5. Check results and finish

Compare temperature and target with the front panel/reference. Confirm the reported operating mode and active alarm names; these flags do not establish physical flow or leak freedom.

On normal completion the summary reports the sample count and confirms that no CSV was written. On an error or kernel interrupt, the controller context closes the worker and serial connection. A forced kernel termination cannot guarantee cleanup. If cleanup fails, allow pending I/O to finish and retry `chiller.disconnect()` rather than creating a competing owner. Use the separate equipment procedure when you intend to stop the chiller or pump.

Next: run `07_dashboard.py` for the browser dashboard.
